# PrismBench Example

End-to-end walkthrough of how a single cell of the PrismBench benchmark is produced. We trace one *(agent, task, run)* triple from configuration to scorecard, then show how 165 such cells aggregate into a routing recommendation.

Steps:

1. Open the task config and the prompt every agent sees.
2. Inspect the dataset that gets staged into the agent workspace.
3. Open one cached *(agent, task, run)* result.
4. Aggregate three runs into mean and CV.
5. Connect the per-cell results to the master scorecard.
6. Feed the master scorecard to the router for a recommendation.
7. (Optional) Re-execute one agent live, if API keys are present.


## 1. The task config

All task definitions live in [`configs/tasks.yaml`](configs/tasks.yaml). Each task carries its own prompt, dataset binding, primary metric, and split seed. We pick `HD-PRED-01` (heart-disease binary classification, 303 rows) for this walkthrough because every agent succeeds on it.


In [1]:
from src.task_runner import load_tasks_config

tasks = load_tasks_config()
task = tasks['HD-PRED-01']
print('task_id:       ', task.get('task_id', 'HD-PRED-01'))
print('dataset:       ', task['dataset'])
print('task_type:     ', task['task_type'])
print('primary metric:', task['primary_metric'])
print('split seed:    ', task.get('split_seed'))


task_id:        HD-PRED-01
dataset:        heart_disease
task_type:      classification
primary metric: f1
split seed:     42


In [2]:
# The exact prompt every agent receives, verbatim. Section 3.3 of the
# report calls this the *same-prompt rule* (Protocol A).
print(task['prompt'])


You are given a CSV file called 'heart_disease.csv' in the current directory.
The target column is 'target' (binary: 0 = no disease, 1 = disease).

Task: Build a binary classifier.
1. Handle any missing values appropriately
2. Encode categorical features if needed
3. Split data 80/20 (use random_state=42)
4. Train at least 3 different classification models
5. Evaluate each on the test set: accuracy, F1-score (weighted), AUC-ROC
6. Select the best model and explain your choice

Save your complete Python code as 'solution.py'.
Save predictions on the test set as 'predictions.csv' with columns: 
'y_true', 'y_pred', 'y_prob' (probability of class 1).
Save metrics as 'metrics.json'.



## 2. The dataset

Before each run, `task_runner` stages the dataset into the agent's per-run workspace. The agent reads the file by name -- it never has to resolve paths.


In [3]:
import pandas as pd
df = pd.read_csv('data/raw/heart_disease.csv')
print(f'shape: {df.shape}')
df.head()


shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## 3. Open one cached result

Every agent run lands in `results/{agent}/{task}/run_{n}/` and contains:

- `result.json`  the raw output dict (predictions, code, tokens, etc.)
- `scorecard.json`  the six-dimension scoring of that run
- `workspace/`  the staged dataset and any agent-generated files

We open `autogluon` on `HD-PRED-01`, run 1 below.


In [4]:
import json
from pathlib import Path

run_dir = Path('results/autogluon/HD-PRED-01/run_1')
print('files in run dir:')
for p in sorted(run_dir.iterdir()):
    print(' ', p.name)


files in run dir:
  autogluon_model
  result.json
  scorecard.json
  workspace


In [5]:
scorecard = json.loads((run_dir / 'scorecard.json').read_text())
print(json.dumps({
    'agent':   scorecard['agent'],
    'task_id': scorecard['task_id'],
    'run_id':  scorecard['run_id'],
    'D1':      scorecard.get('D1_accuracy'),
    'D2':      scorecard.get('D2_code_quality'),
    'D3':      scorecard.get('D3_explainability'),
    'D4':      scorecard.get('D4_speed'),
    'D5':      scorecard.get('D5_cost'),
    'D6':      scorecard.get('D6_robustness'),
}, indent=2, default=str))


{
  "agent": "autogluon",
  "task_id": "?",
  "run_id": 1,
  "D1": {
    "type": "classification",
    "f1_weighted": 0.9017,
    "accuracy": 0.9016,
    "auc_roc": 0.9569
  },
  "D2": {
    "pylint_score": 0.0,
    "llm_judge_score": null,
    "combined": 0.0,
    "note": "No code generated by agent"
  },
  "D3": {
    "shap_generated": false,
    "feature_importance_generated": false,
    "has_explanation_text": true,
    "auto_score": 3.0,
    "llm_judge_score": null
  },
  "D4": {
    "wall_clock_sec": 122.054
  },
  "D5": {
    "api_cost_usd": 0.0,
    "input_tokens": 0,
    "output_tokens": 0,
    "tokens_total": 0,
    "carbon_kg": 0.00035853,
    "carbon_source": "measured"
  },
  "D6": null
}


## 4. Aggregate three runs

Each (agent, task) cell is run three times. The robustness dimension `D6` is the coefficient of variation of `D1` across those runs -- a consistent agent has low CV.


In [6]:
import numpy as np

f1_scores = []
for n in (1, 2, 3):
    sc = json.loads((Path(f'results/autogluon/HD-PRED-01/run_{n}/scorecard.json')).read_text())
    d1 = sc.get('D1_accuracy', {}) or {}
    f1 = d1.get('f1_weighted')
    f1_scores.append(f1)
    print(f'run {n}: F1 = {f1}')

f1_arr = np.array([s for s in f1_scores if s is not None], dtype=float)
print(f'\nmean F1: {f1_arr.mean():.4f}')
print(f'std F1:  {f1_arr.std(ddof=0):.4f}')
if f1_arr.mean() > 0:
    print(f'CV (D6): {f1_arr.std(ddof=0) / f1_arr.mean():.4f}')


run 1: F1 = 0.9017
run 2: F1 = 0.9017
run 3: F1 = 0.9017

mean F1: 0.9017
std F1:  0.0000
CV (D6): 0.0000


## 5. Connect to the master scorecard

`src.scorecard.build_scorecard()` walks `results/` and produces `results/master_scorecard.csv` (one row per run) and `results/master_scorecard_summary.csv` (one row per *(agent, task)* with the mean across runs). The summary CSV is what the router consumes.


In [7]:
master = pd.read_csv('results/master_scorecard.csv')
print(f'total rows: {len(master)}')

ag = master[(master.agent == 'autogluon') & (master.task_id == 'HD-PRED-01')]
ag[['agent', 'task_id', 'run_id', 'D1_f1', 'D4_time_sec', 'D5_cost_usd', 'D5_carbon_kg']]


total rows: 198


,agent,task_id,run_id,D1_f1,D4_time_sec,D5_cost_usd,D5_carbon_kg
77,autogluon,HD-PRED-01,1,0.9017,122.054,0.0,0.000359
78,autogluon,HD-PRED-01,2,0.9017,121.346,0.0,0.000355
79,autogluon,HD-PRED-01,3,0.9017,122.213,0.0,0.000358


## 6. Feed the router

The router ingests the summary CSV and returns a top-1 agent under each MCDM method, given a weight vector. We try the `accuracy` preset (60% on D1) and the `frugal` preset (30% cost, 20% carbon).


In [8]:
from src import router

for preset_name in ['balanced', 'accuracy', 'frugal']:
    rec = router.recommend(router.PRESETS[preset_name])
    print(f'\n=== preset: {preset_name} ===')
    for method, scores in rec['rankings'].items():
        top3 = list(scores.index[:3])
        print(f'  {method:>10}: {top3}')



=== preset: balanced ===
         wsm: ['claude_code', 'langgraph', 'autogluon']
      topsis: ['claude_code', 'langgraph', 'chatgpt_ada']
   promethee: ['claude_code', 'langgraph', 'autogluon']

=== preset: accuracy ===
         wsm: ['autogluon', 'claude_code', 'langgraph']
      topsis: ['autogluon', 'langgraph', 'claude_api_raw']
   promethee: ['autogluon', 'claude_code', 'langgraph']

=== preset: frugal ===
         wsm: ['claude_code', 'autogluon', 'langgraph']
      topsis: ['claude_code', 'autogluon', 'langgraph']
   promethee: ['claude_code', 'autogluon', 'langgraph']


## 7. (Optional) Live run

The cells above use cached results so this notebook executes in seconds. If you want to see the loop run live, the cell below executes `autogluon` on a *fresh* run. It takes about 90-120 seconds and writes to `results/autogluon/HD-PRED-01/run_99/`.

Skip this cell during grading review if you don't want to wait.


In [9]:
# Uncomment to run live. Requires the autogluon Python package; no API
# key needed.
#
# from src.task_runner import run_single
# result = run_single('autogluon', 'HD-PRED-01', run_id=99)
# print({k: result[k] for k in ('agent_id', 'task_id', 'wall_clock_sec', 'error')})


## Where to go next

- [`prismbench.API.ipynb`](prismbench.API.ipynb)  pure-API tour of the router.
- [`report.pdf`](report.pdf)  full study, including the multi-protocol analysis and Theoretical Proposition 1.
- `python -m src.run_benchmark --pilot`  three agents on two tasks, single run; ten minutes end to end.
- `python -m src.run_benchmark`  full 81-cell main benchmark; an hour or so, depending on API rate limits.
